# Colab Encoder Run: XLM-RoBERTa & MuRIL Evaluation

This notebook runs the frozen transformer encoder baselines (`xlm-roberta-base` and `google/muril-base-cased`) for the CJP Protest Disinformation Triage study.

> **MANDATORY NOTICE & WARNING:**
> - Feasibility pilot on documented claim records; scores are unstable and are not evidence that AI propaganda can be detected.
> - Synthetic / demo outputs must NEVER be cited or placed into `outputs/` or the final report.
> - If Hugging Face model downloads stall, Colab GPU runtimes provide dedicated bandwidth and CUDA acceleration.

In [ ]:
# Pinned dependencies for reproducible execution
!pip install --upgrade pip
!pip install torch==2.3.0 transformers==4.40.0 scikit-learn==1.4.2 pandas==2.2.2 numpy==1.26.4 joblib==1.4.2 openpyxl==3.1.2

In [ ]:
import os
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active device: {device}")

In [ ]:
# Load final_labels.csv and narrative_mapping.csv
# Ensure data/processed/final_labels.csv and narrative_mapping.csv are uploaded
data_path = Path("data/processed/final_labels.csv")
if not data_path.exists():
    raise FileNotFoundError("Please upload final_labels.csv to data/processed/")

df = pd.read_csv(data_path)
mapping_path = Path("narrative_mapping.csv")
mapping = pd.read_csv(mapping_path)

cat_map = {r['native_label']: r['proposal_code'] for _, r in mapping[mapping['level'] == 'category'].iterrows()}
row_map = {rid.strip(): r['proposal_code'] for _, r in mapping[mapping['level'] == 'row'].iterrows() for rid in str(r['record_ids']).split(';')}

def get_scheme_p(r):
    if str(r['record_id']).strip() in row_map:
        return row_map[str(r['record_id']).strip()]
    return cat_map.get(str(r['narrative_native']).strip(), 'other')

df['scheme_p_raw'] = df.apply(get_scheme_p, axis=1)
claims_p = df.groupby('scheme_p_raw')['claim_id'].nunique()
rare = set(claims_p[claims_p < 8].index)
df['scheme_p'] = df['scheme_p_raw'].apply(lambda x: 'other' if x in rare else x)

print(f"Loaded {len(df)} records ({df['claim_id'].nunique()} distinct claims) across {df['scheme_p'].nunique()} Scheme P classes.")

In [ ]:
# Compute or cache embeddings
models = {
    "xlmr": "xlm-roberta-base",
    "muril": "google/muril-base-cased",
}
embeddings = {}

texts = df['claim_text'].tolist()
for key, hf_name in models.items():
    print(f"Extracting embeddings for {hf_name}...")
    tok = AutoTokenizer.from_pretrained(hf_name)
    model = AutoModel.from_pretrained(hf_name).to(device)
    model.eval()
    
    all_embs = []
    batch_size = 16
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tok(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model(**enc)
            mask = enc['attention_mask'].unsqueeze(-1).expand(out.last_hidden_state.size()).float()
            pooled = (out.last_hidden_state * mask).sum(dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)
            all_embs.append(pooled.cpu().numpy())
    embeddings[key] = np.vstack(all_embs)
    print(f"Shape for {key}: {embeddings[key].shape}")

In [ ]:
# Run 5-seed 5-fold StratifiedGroupKFold CV
CV_SEEDS = [42, 43, 44, 45, 46]
labels = df['scheme_p'].tolist()
groups = df['claim_id'].tolist()

for key, X in embeddings.items():
    scores = []
    for seed in CV_SEEDS:
        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        for train_idx, test_idx in sgkf.split(texts, labels, groups=groups):
            sc = StandardScaler()
            X_tr = sc.fit_transform(X[train_idx])
            X_te = sc.transform(X[test_idx])
            clf = LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0, random_state=seed)
            clf.fit(X_tr, [labels[i] for i in train_idx])
            preds = clf.predict(X_te)
            scores.append(f1_score([labels[i] for i in test_idx], preds, average='macro', zero_division=0))
    print(f"{key}_lr Scheme P: Mean Macro-F1 = {np.mean(scores):.4f} +/- {np.std(scores):.4f}")

### Rebuilding Report Locally
After running this notebook and copying updated outputs to `outputs/`, simply rebuild the report with:
```bash
python build_report.py
```
No manual rewriting of `REPORT.md` is required.